# Job Lifecycle Fact Table Loader

This notebook maintains the `warehouse.fact_job_lifecycle` fact table.

**Purpose**: Track job state transitions and lifecycle events

**Key Features**:
* One grain: One row per job state change event
* Captures transition types (new→active, active→inactive, inactive→deleted, etc.)
* Duration metrics between states
* SCD2-aware dimension joins

**Sources**: 
* `silver.silver_job_changes` (state change events)
* `warehouse.dim_job` (job context)
* Other dimension tables

**Target**: `workspace.warehouse.fact_job_lifecycle`

In [0]:
%sql
-- Extract job lifecycle change events
CREATE OR REPLACE TEMP VIEW lifecycle_events AS
SELECT 
  c.change_id,
  c.enterprise_job_id,
  c.source_name,
  c.change_type,
  c.changed_columns,
  c.change_timestamp,
  c.batch_id,
  -- Parse old and new state from JSON
  GET_JSON_OBJECT(c.old_value_json, '$.is_active') as old_is_active,
  GET_JSON_OBJECT(c.new_value_json, '$.is_active') as new_is_active,
  GET_JSON_OBJECT(c.old_value_json, '$.soft_delete_flag') as old_soft_delete,
  GET_JSON_OBJECT(c.new_value_json, '$.soft_delete_flag') as new_soft_delete
FROM workspace.silver.silver_job_changes c
WHERE c.enterprise_job_id IS NOT NULL
  AND c.change_type IS NOT NULL;

In [0]:
%sql
-- Classify lifecycle transition types
CREATE OR REPLACE TEMP VIEW lifecycle_classified AS
SELECT 
  e.*,
  CASE 
    WHEN e.change_type = 'INSERT' THEN 'JOB_CREATED'
    WHEN e.change_type = 'DELETE' THEN 'JOB_DELETED'
    WHEN e.old_is_active = 'false' AND e.new_is_active = 'true' THEN 'JOB_ACTIVATED'
    WHEN e.old_is_active = 'true' AND e.new_is_active = 'false' THEN 'JOB_DEACTIVATED'
    WHEN e.old_soft_delete = 'false' AND e.new_soft_delete = 'true' THEN 'JOB_SOFT_DELETED'
    WHEN e.old_soft_delete = 'true' AND e.new_soft_delete = 'false' THEN 'JOB_RESTORED'
    WHEN e.change_type = 'UPDATE' THEN 'JOB_UPDATED'
    ELSE 'UNKNOWN_TRANSITION'
  END as lifecycle_event_type
FROM lifecycle_events e;

In [0]:
%sql
-- Calculate time since last change (duration in current state)
CREATE OR REPLACE TEMP VIEW lifecycle_with_duration AS
SELECT 
  l.*,
  LAG(l.change_timestamp) OVER (
    PARTITION BY l.enterprise_job_id 
    ORDER BY l.change_timestamp
  ) as previous_change_timestamp,
  DATEDIFF(
    l.change_timestamp,
    LAG(l.change_timestamp) OVER (
      PARTITION BY l.enterprise_job_id 
      ORDER BY l.change_timestamp
    )
  ) as days_in_previous_state
FROM lifecycle_classified l;

In [0]:
%sql
-- Resolve foreign keys using SCD2 time travel
CREATE OR REPLACE TEMP VIEW fact_lifecycle_with_fks AS
SELECT 
  l.*,
  -- Job SK - resolve to version effective at change time
  COALESCE(j.job_sk, -1) as job_sk,
  COALESCE(j.company_sk, -1) as company_sk,
  COALESCE(j.location_sk, -1) as location_sk,
  COALESCE(j.sector_sk, -1) as sector_sk,
  -- Role SK
  COALESCE(dr.role_sk, -1) as role_sk,
  -- Source SK
  COALESCE(ds.source_sk, -1) as source_sk,
  -- Change date SK
  CAST(DATE_FORMAT(l.change_timestamp, 'yyyyMMdd') AS INT) as change_date_sk
FROM lifecycle_with_duration l
-- SCD2 join to job dimension
LEFT JOIN workspace.warehouse.dim_job j
  ON l.enterprise_job_id = j.enterprise_job_id
  AND l.change_timestamp >= j.effective_from
  AND (l.change_timestamp < j.effective_to OR j.effective_to IS NULL)
-- Role dimension
LEFT JOIN workspace.warehouse.dim_role dr
  ON j.canonical_role_id = dr.canonical_role_id
-- Source dimension
LEFT JOIN workspace.warehouse.dim_source ds
  ON l.source_name = ds.source_name;

In [0]:
%sql
-- Merge into fact table
MERGE INTO workspace.warehouse.fact_job_lifecycle target
USING (
  SELECT
    COALESCE(f.fact_job_lifecycle_sk,
      ROW_NUMBER() OVER (ORDER BY fk.change_id) + COALESCE(max_sk, 0)) as fact_job_lifecycle_sk,
    fk.job_sk,
    fk.company_sk,
    fk.location_sk,
    fk.role_sk,
    fk.sector_sk,
    fk.source_sk,
    fk.change_date_sk,
    fk.change_timestamp,
    fk.change_type,
    fk.lifecycle_event_type,
    fk.days_in_previous_state,
    fk.batch_id
  FROM fact_lifecycle_with_fks fk
  LEFT JOIN workspace.warehouse.fact_job_lifecycle f
    ON fk.change_id = f.change_id
  CROSS JOIN (
    SELECT COALESCE(MAX(fact_job_lifecycle_sk), 0) as max_sk 
    FROM workspace.warehouse.fact_job_lifecycle
  ) max_keys
  WHERE fk.job_sk IS NOT NULL AND fk.job_sk != -1
) source
ON target.change_id = source.change_id
WHEN NOT MATCHED THEN INSERT (
  fact_job_lifecycle_sk,
  job_sk,
  company_sk,
  location_sk,
  role_sk,
  sector_sk,
  source_sk,
  change_date_sk,
  change_timestamp,
  change_type,
  lifecycle_event_type,
  days_in_previous_state,
  batch_id
) VALUES (
  source.fact_job_lifecycle_sk,
  source.job_sk,
  source.company_sk,
  source.location_sk,
  source.role_sk,
  source.sector_sk,
  source.source_sk,
  source.change_date_sk,
  source.change_timestamp,
  source.change_type,
  source.lifecycle_event_type,
  source.days_in_previous_state,
  source.batch_id
);

In [0]:
%sql
-- Validate lifecycle fact table
SELECT 
  COUNT(*) as total_lifecycle_events,
  COUNT(DISTINCT job_sk) as jobs_with_changes,
  lifecycle_event_type,
  COUNT(*) as event_count,
  AVG(days_in_previous_state) as avg_days_before_change
FROM workspace.warehouse.fact_job_lifecycle
GROUP BY lifecycle_event_type
ORDER BY event_count DESC;

-- Sample lifecycle events
SELECT 
  f.fact_job_lifecycle_sk,
  j.enterprise_job_id,
  j.title_normalized,
  c.company_name,
  f.change_timestamp,
  f.lifecycle_event_type,
  f.days_in_previous_state
FROM workspace.warehouse.fact_job_lifecycle f
INNER JOIN workspace.warehouse.dim_job j ON f.job_sk = j.job_sk
INNER JOIN workspace.warehouse.dim_company c ON f.company_sk = c.company_sk
ORDER BY f.change_timestamp DESC
LIMIT 20;